In [1]:
import pandas as pd

# FAOSTAT API DS query
# Địa chỉ endpoint FAOSTAT JSON
url = "https://api.fao.org/api/3/agriculture/Domain/Production?"

# Tham số query
query = {
    "area": "VNM",  # Vietnam
    "item": "Rice, paddy",  # gạo nguyên liệu
    "yearRange": "2010:2023",
    "output": "json"
}

import requests
response = requests.get(url, params=query)

# --- Added for debugging ---
print(f"Response Status Code: {response.status_code}")
print(f"Response Text: {response.text}")
# --- End debugging ---

# Check if the request was successful before trying to parse JSON
if response.status_code == 200:
    data = response.json()

    # Chuyển sang DataFrame
    df_fao = pd.json_normalize(data, record_path=['data'])

    # Lọc cột cần thiết
    df_fao = df_fao[['Year', 'Value', 'elementCode', 'element']]

    # Pivot cho dễ dùng
    df_fao_pivot = df_fao.pivot_table(
        index="Year",
        columns="element",
        values="Value"
    ).reset_index()

    df_fao_pivot = df_fao_pivot.rename(columns={
        "Production": "Production_tons",
        "Area harvested": "AreaHarvested_ha",
        "Yield per hectare": "Yield_t_per_ha"
    })

    # Xuất JSON
    df_fao_pivot.to_json(
        "faostat_rice_vietnam.json",
        orient="records",
        force_ascii=False,
        indent=4
    )

    print("DONE: FAOSTAT JSON created!")
else:
    print(f"Error: API returned status code {response.status_code}")
    print(f"Response content: {response.text}")
    print("Please check the API URL and parameters. The API endpoint might be incorrect or unavailable.")

Response Status Code: 404
Response Text: <!DOCTYPE HTML PUBLIC "-//IETF//DTD HTML 2.0//EN">
<html><head>
<title>404 Not Found</title>
</head><body>
<h1>Not Found</h1>
<p>The requested URL /api/3/agriculture/Domain/Production was not found on this server.</p>
</body></html>

Error: API returned status code 404
Response content: <!DOCTYPE HTML PUBLIC "-//IETF//DTD HTML 2.0//EN">
<html><head>
<title>404 Not Found</title>
</head><body>
<h1>Not Found</h1>
<p>The requested URL /api/3/agriculture/Domain/Production was not found on this server.</p>
</body></html>

Please check the API URL and parameters. The API endpoint might be incorrect or unavailable.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')